# Principled Comparison of Robot Policy Performance

We illustrate the use of statistically rigorous sequential evaluation procedures to rapidly and reliably compare robot policy performance

### Import Lightweight Python Libraries

In [2]:
import numpy as np 
import os 
import sys
from tqdm import tqdm 

### Import Rigorous Policy Comparison Methods

In [3]:
# General tools for statistical hypothesis testing
from sequentialized_barnard_tests.base import Hypothesis, Decision

# Tools specifically for comparison under binary performance measures
from sequentialized_barnard_tests.step import StepTest
from sequentialized_barnard_tests.savi import SaviTest

#####
# New, general-purpose tools for arbitrary, bounded performance measures
#####
path_for_loading_nscore = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(path_for_loading_nscore)

# WSR METHOD: Unstructured nonparametric method (more general, but less sample efficient)
from nscore.wsr import WsrComparisonTest

# Theta-SAVI METHOD: Explicitly structured parametric method (less general, but more sample efficient)
from nscore.savi import PartialCreditSaviTest

# Our method (NSCORE): as general as WSR and as efficient as Theta-SAVI 
from nscore.nsm import BernoulliNsmTest
from nscore.nonparametric_nsm import ContinuousNsmTest

## The Setting of Policy Comparison

Recall (see [here](https://medium.com/toyotaresearch/statistical-thinking-for-robot-policy-evaluation-from-rigorous-a-b-testing-to-effective-0ae886fbd68d) for more details) that we are considering the problem of proving, at given confidence level $1 - \alpha$, that the mean performance of our robot policy is better than a baseline (i.e., the current state of the art). Because robot evaluation is expensive, we want to do this as quickly as possible. 

Mathematically, we will express the comparison in the following terms: 
-  $\mu_1 = \mu(\pi_1)$, the mean performance of the new policy
-  $\mu_0 = \mu(\pi_0)$, the mean performance of the state-of-the-art baseline

The question is precisely: determine if, given the available data, $\mu_1 > \mu_0 \text{ w.p.} \geq 1-\alpha$. 

However, the desiderata of our determination -- high confidence and small sample size -- are in conflict! Concluding that our new policy is better with high confidence requires us to be very careful in how we use the limited data at our disposal. 

Currently, the standard evaluation practice is to choose a batch size, $N$, run $N$ evaluations of each policy, and report the change in empirical performance. However, this can be very misleading, as the following example illustrates. 

### A Simple Mental Experiment: The "Evaluation Code Bug"

Imagine that you are pushing for a deadline. Though it'll be tight, you're almost there: all you have left to do is run the hardware evaluations of your new policy against the current state of the art baseline. All of the stuff is ready (after much suffering, of course), but in the last days of the crunch, everything goes without a hitch. 

There's just one problem -- a silent bug. Specifically: the variable loading the policy weights was inadvertantly hardcoded from an earlier debugging phase, and every time an evaluation was run, the baseline policy was loaded!

Imagine the actual data collection in that case: 
- For the `baseline data,' the true mean is $\mu_0$. 
- For the `new policy,' the true mean is ... also $\mu_0$!

What happens downstream? Well, let's imagine that the performance measure is binary (success / failure), and assume that 25 evaluations were run per policy. What is the chance of observing at least 8 percentage points of improvement empirically?

I strongly encourage the interested reader to formulate a concrete estimate. For context: the correct answer will be somewhere between 0% and 50%; we will assume that the true baseline policy success rate is 57% (to choose a random prime number...). The answer will be printed at the end of the following code snippet. Additionally, note that the random seed can be freely adjusted to verify that the number is not itself a fluke result. 

In [5]:
n_fictitious_evaluation_sequences = 10000
n_evaluations_per_sequence = 25 

p0 = 0.57       # Baseline succeeds 57% of the time
p1 = 0.57       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get at least 8 percentage point improvement from random noise. 
eight_pp_improvement = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(42)

for i in range(n_fictitious_evaluation_sequences):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    if np.mean(data_new_policy) >= (np.mean(data_baseline) + 0.08 - 1e-8):
        eight_pp_improvement[i] += 1.0 


print(f"Percentage of cases where we observe 8 percentage point improvement from random noise: {100.*np.mean(eight_pp_improvement):0.2f}%")

Percentage of cases where we observe 8 percentage point improvement from random noise: 33.77%


### Completing the Idea: Comparison Without Software Bugs

Here is the key context for applying the thought experiment to practical evaluation: when we make changes to policy architectures, training datasets, hyperparameters, etc, we __might not be improving the overall performance__. The systems we are engaging with are profoundly complex. As such, the sum of our innovations in a particular phase of a project may be the tragic tale of difficult research: a sound and fury, signifying nothing. 

As such, even when we avoid the bug described in the preceding section, we must still imagine that the new policy, while qualitatively different than the baseline, may still not have improved in average performance. We might be exactly in the realm of the preceding section! 

### Empirical Performance Gaps are Not Enough at Small Sample Sizes
